In [1]:
import json
import duckdb

# 连接你本地的 DuckDB 数据库
con = duckdb.connect("kdrama_reddit.duckdb", read_only=True)

# 对应 extract_thread_bodies.py 里的 5 个文件名和剧名关键词
drama_files = {
    "threads_mask_girl.jsonl": "mask girl",
    "threads_true_beauty.jsonl": "true beauty",
    "threads_my_id_is_gangnam_beauty.jsonl": "my id is gangnam beauty",
    "threads_birth_of_a_beauty.jsonl": "birth of a beauty",
    "threads_boys_over_flowers.jsonl": "boys over flowers",
}

for filename, keyword in drama_files.items():
  print(f"正在检索并生成 {filename} ...")

  # 复刻网页版的 "Search Any Comment in Thread" 逻辑：
  # 找出主帖(title/selftext)或评论(comment_body)中包含该剧名的所有 unique post_id
  query = f"""
        SELECT DISTINCT post_id FROM (
            SELECT post_id 
            FROM submissions 
            WHERE LOWER(title) LIKE '%{keyword}%' OR LOWER(selftext) LIKE '%{keyword}%'
            
            UNION
            
            SELECT (CASE WHEN post_id LIKE 't3_%' THEN SUBSTR(post_id, 4) ELSE post_id END) AS post_id
            FROM comments 
            WHERE LOWER(comment_body) LIKE '%{keyword}%'
        )
    """
  df_ids = con.execute(query).fetchdf()

  # 写入 .jsonl 文件，格式完全符合 extract_thread_bodies.py 的读取要求
  with open(filename, "w", encoding="utf-8") as f:
    for pid in df_ids["post_id"]:
      if pid:
        f.write(json.dumps({"post_id": f"t3_{pid}"}) + "\n")

  print(f"  -> 已保存 {len(df_ids)} 个 thread IDs 到 {filename}")

con.close()
print("\n✅ 5 个前置 .jsonl 文件已全部生成完毕！")

正在检索并生成 threads_mask_girl.jsonl ...
  -> 已保存 1696 个 thread IDs 到 threads_mask_girl.jsonl
正在检索并生成 threads_true_beauty.jsonl ...
  -> 已保存 10742 个 thread IDs 到 threads_true_beauty.jsonl
正在检索并生成 threads_my_id_is_gangnam_beauty.jsonl ...
  -> 已保存 4359 个 thread IDs 到 threads_my_id_is_gangnam_beauty.jsonl
正在检索并生成 threads_birth_of_a_beauty.jsonl ...
  -> 已保存 1126 个 thread IDs 到 threads_birth_of_a_beauty.jsonl
正在检索并生成 threads_boys_over_flowers.jsonl ...
  -> 已保存 7957 个 thread IDs 到 threads_boys_over_flowers.jsonl

✅ 5 个前置 .jsonl 文件已全部生成完毕！
